<a href="https://colab.research.google.com/github/Limeng-svg/Grounded-PPE-Safety-Copilot/blob/main/notebooks/03_yoloworld_prompt_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This round will fix the model, images, annotations, and evaluation parameters, only changing one PPE prompt, and will automatically generate comparison results and a download package.

In [3]:
%pip -q install "ultralytics==8.4.129" "git+https://github.com/ultralytics/CLIP.git@68dce32140994dfcb645a1320c4ebdc034fc19fd"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00


In [5]:
from google.colab import drive
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import tempfile, shutil, json

import yaml
import torch
import ultralytics
import pandas as pd
import matplotlib.pyplot as plt

drive.mount("/content/drive")

assert ultralytics.__version__ == "8.4.129", \
    "Please restart the Colab session and run this cell again."
assert torch.cuda.is_available(), \
    "Please set the hardware accelerator to T4 GPU."

print("GPU:", torch.cuda.get_device_name(0))

ROOT = Path(
    "/content/drive/MyDrive/Grounded-PPE-Safety-Copilot"
)
SOURCE = ROOT / "data/mini_eval"

CLASS_NAMES = [
    "person",
    "hard_hat",
    "safety_vest",
]

EXPECTED_COUNTS = Counter({
    0: 60,
    1: 53,
    2: 37,
})

images = sorted((SOURCE / "images").glob("*.jpg"))

expected_stems = {
    f"{group}_{i:02d}"
    for group in ["easy", "hard", "violation"]
    for i in range(1, 5)
}

assert len(images) == 12, "Expected 12 images but found fewer. Please check the data path and contents."
assert {p.stem for p in images} == expected_stems

# Copy data to a temporary Colab directory for faster read/write access;
# This will not modify the original images and labels in Google Drive.
LOCAL = Path(
    tempfile.mkdtemp(
        prefix="ppe_prompt_ablation_",
        dir="/content",
    )
)

for folder in ["images/val", "labels/val"]:
    (LOCAL / folder).mkdir(parents=True)

counts = Counter()

for image in images:
    label = SOURCE / "labels" / f"{image.stem}.txt"
    assert label.is_file(), f"Missing label file: {label.name}"

    for row in label.read_text(
        encoding="utf-8"
    ).splitlines():
        if row.strip():
            parts = row.split()
            assert len(parts) == 5, \
                f"Incorrect label format: {label.name}"
            counts[int(parts[0])] += 1

    shutil.copy2(
        image,
        LOCAL / "images/val" / image.name,
    )
    shutil.copy2(
        label,
        LOCAL / "labels/val" / label.name,
    )

assert counts == EXPECTED_COUNTS, \
    f"Label count mismatch: {counts}"

print("Data preparation complete: 12 images, 150 ground truth bounding boxes.")

Mounted at /content/drive
GPU: Tesla T4
Data preparation complete: 12 images, 150 ground truth bounding boxes.
